## PCA sandbox

Pulls small samples of the raw Kaggle datasets into `data/sandbox/` and runs the PCA comparison end-to-end as a quick smoke test.

In [1]:
import kagglehub

# Stock data
path = kagglehub.dataset_download("jacksoncrow/stock-market-dataset")
print("Path to dataset files:", path)

100%|██████████| 522M/522M [00:33<00:00, 16.1MB/s] 

Extracting files...


Path to dataset files: ~/.cache/kagglehub/datasets/jacksoncrow/stock-market-dataset/versions/2


In [2]:
#Poly data
path = kagglehub.dataset_download("ismetsemedov/polymarket-prediction-markets")
print("Path to dataset files:", path)

100%|██████████| 42.4M/42.4M [00:05<00:00, 8.37MB/s]

Extracting files...


Path to dataset files: ~/.cache/kagglehub/datasets/ismetsemedov/polymarket-prediction-markets/versions/1


In [4]:
from pathlib import Path
import pandas as pd

repo_root = Path.cwd().parent
stock_src = Path.home() / Path(".cache/kagglehub/datasets/jacksoncrow/stock-market-dataset/versions/2/stocks/AAPL.csv")
poly_src = Path.home() / Path(".cache/kagglehub/datasets/ismetsemedov/polymarket-prediction-markets/versions/1/polymarket_markets.csv")

stock_out = repo_root / "data/sandbox/stocks/demo_stock_aapl.csv"  # small sandbox sample used to smoke-test the PCA pipeline
poly_out = repo_root / "data/sandbox/polymarket/demo_polymarket_markets.csv"

stock_out.parent.mkdir(parents=True, exist_ok=True)
poly_out.parent.mkdir(parents=True, exist_ok=True)

stock_df = pd.read_csv(stock_src).head(300)
stock_df.to_csv(stock_out, index=False)

poly_cols_preferred = [
    "id",
    "slug",
    "bestAsk",
    "bestBid",
    "liquidity",
    "liquidityAmm",
    "liquidityClob",
    "volume",
    "volumeNum",
    "volume24hr",
    "volume24hrAmm",
    "volume24hrClob",
    "oneDayPriceChange",
    "oneWeekPriceChange",
    "oneMonthPriceChange",
    "oneYearPriceChange",
    "spread",
    "orderMinSize",
    "orderPriceMinTickSize",
    "marketType",
    "sportsMarketType",
    "category",
    "subcategory",
]

poly_head = pd.read_csv(poly_src, nrows=0)
poly_cols = [col for col in poly_cols_preferred if col in poly_head.columns]
poly_df = pd.read_csv(poly_src, usecols=poly_cols).head(500)
poly_df.to_csv(poly_out, index=False)

print(f"Wrote {len(stock_df)} rows -> {stock_out}")
print(f"Wrote {len(poly_df)} rows -> {poly_out}")
print(f"Polymarket columns used ({len(poly_cols)}): {poly_cols}")

Wrote 300 rows -> data/sandbox/stocks/demo_stock_aapl.csv
Wrote 500 rows -> data/sandbox/polymarket/demo_polymarket_markets.csv
Polymarket columns used (23): ['id', 'slug', 'bestAsk', 'bestBid', 'liquidity', 'liquidityAmm', 'liquidityClob', 'volume', 'volumeNum', 'volume24hr', 'volume24hrAmm', 'volume24hrClob', 'oneDayPriceChange', 'oneWeekPriceChange', 'oneMonthPriceChange', 'oneYearPriceChange', 'spread', 'orderMinSize', 'orderPriceMinTickSize', 'marketType', 'sportsMarketType', 'category', 'subcategory']


/var/folders/tj/3jft9cyd7gbbzj2vmd9982yw0000gn/T/ipykernel_50506/4273431072.py:45: DtypeWarning: Columns (9,64,99,103) have mixed types. Specify dtype option on import or set low_memory=False.
  poly_df = pd.read_csv(poly_src, usecols=poly_cols).head(500)


In [ ]:
import json
import sys
from pathlib import Path

repo_root = Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from polystock.dimensionality import compare_domains_pca

result = compare_domains_pca(
    stock_csv_path=repo_root / "data/sandbox/stocks/demo_stock_aapl.csv",
    polymarket_csv_path=repo_root / "data/sandbox/polymarket/demo_polymarket_markets.csv",
    variance_threshold=0.95,
    scaler="standard",
    stock_drop_cols=["Date"],
    polymarket_drop_cols=["id", "slug", "marketType", "sportsMarketType", "category", "subcategory"],
)

print(json.dumps(result["comparison"], indent=2))
print("stock components to 95%:", result["stock"]["components_to_threshold"])
print("polymarket components to 95%:", result["polymarket"]["components_to_threshold"])
